# E1.10 · The stakeholder map: who owns what

**Function E — AI for GRC → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

Builds on **[E1.9 · Model and agent lifecycle governance](https://spbreed.github.io/cyber-commons/lessons/E1.9.html)**.

| | |
|---|---|
| Open-source tooling | NIST AI RMF, ISO 42001 |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The hook

Legal, privacy, model risk and security each hold a piece of the AI control estate, and none of them holds all of it. Every failure in this function is a failure at a boundary between two of those teams.

## 2 · The framework

```
   the AI control estate, and who holds a piece of it

   +----------+ +---------+ +--------------+ +----------+ +---------+
   |  legal   | | privacy | | model risk   | | security | | product |
   +----------+ +---------+ +--------------+ +----------+ +---------+
        \_____________\________|________/_____________/
                          the seams
   every failure in this function happens at a boundary, not inside a box
```

Five functions hold the AI control estate between them, and **none of them holds
all of it.** The programme does not fail inside any one function. It fails at the
seams, where each side reasonably believed the other had it.

| Stakeholder | The question they are actually asking |
|---|---|
| **Legal** | Can we be held liable, and under what theory? |
| **Compliance** | Which obligations apply, and can we demonstrate we meet them? |
| **Data Privacy** | Whose data is in this, on what basis, and for how long? |
| **Cyber Security** | Can this be attacked, and can we contain it if it is? |
| **Model Risk** | Is this fit for its stated purpose, and will we know when it stops being so? |

Two seats are routinely forgotten. The **business or product owner** in the
first line, who defines intended purpose and risk appetite and funds
remediation — if that seat is empty, the other five are governing an orphan. And
**internal audit** in the third line, whose job is independent assurance that the
five are doing what they claim.

What makes this a lesson rather than an org chart is the four gaps below. Each
one is a real failure that happens because *both* sides made a reasonable
assumption about the other.

## 3 · Who operates which control

In [ ]:
STAKEHOLDERS = {
 "legal":      {"asks": "can we be held liable, and under what theory",
                "controls": ["contract clauses", "acceptable-use terms",
                             "IP screening", "e-discovery retention"]},
 "compliance": {"asks": "which obligations apply, can we demonstrate we meet them",
                "controls": ["AI policy", "use-case classification",
                             "attestations", "disclosure triggers"]},
 "privacy":    {"asks": "whose data, on what basis, for how long",
                "controls": ["impact assessment gate", "PII redaction",
                             "retention schedules", "transfer mechanisms"]},
 "cyber":      {"asks": "can this be attacked, can we contain it",
                "controls": ["agent identity and JIT authz", "tool permissions",
                             "sandbox and egress", "guardrails",
                             "telemetry and detections", "kill switch"]},
 "model_risk": {"asks": "is it fit for purpose, will we know when it stops being",
                "controls": ["pre-deployment validation", "performance thresholds",
                             "drift alerting", "revalidation on change"]},
}
ALSO = {"business_owner": "accountable for the use case; defines purpose and risk appetite",
        "internal_audit": "independent assurance that the five do what they claim"}

for name in sorted(STAKEHOLDERS):
    s = STAKEHOLDERS[name]
    print(f"{name:12s}{s['asks']}")
    print(f"            controls: {', '.join(s['controls'])}")
print()
for name, role in sorted(ALSO.items()):
    print(f"{name:16s}{role}")
total = sum(len(s["controls"]) for s in STAKEHOLDERS.values())
print(f"\n{total} controls across {len(STAKEHOLDERS)} functions")

## 4 · The four seams, each with two reasonable assumptions

In [ ]:
SEAMS = [
 {"gap": "agent traces are full of personal data",
  "a": ("cyber", "privacy owns retention of anything containing personal data"),
  "b": ("privacy", "security owns the log store, so security sets its schedule"),
  "result": "no schedule was set; three years of prompts are discoverable"},
 {"gap": "the model was validated, the tools were not",
  "a": ("model_risk", "validation covered the model, which is our scope"),
  "b": ("cyber", "MRM signed it off, so the deployment was approved"),
  "result": "an agent holds production write access that was never in scope"},
 {"gap": "'no training on our data' was negotiated, never instrumented",
  "a": ("legal", "the clause is in the contract and it is binding"),
  "b": ("cyber", "legal handled the vendor, so the restriction is handled"),
  "result": "nobody built the control that verifies the vendor honours it"},
 {"gap": "the use case was risk-tiered before it had tools",
  "a": ("compliance", "classified low-risk: it was a chatbot when we saw it"),
  "b": ("business_owner", "we shipped features, not a new use case"),
  "result": "it files tickets, sends mail and moves money at the low-risk tier"},
]
for i, s in enumerate(SEAMS, 1):
    print(f"{i}. {s['gap']}")
    print(f"   {s['a'][0]:15s} assumed: {s['a'][1]}")
    print(f"   {s['b'][0]:15s} assumed: {s['b'][1]}")
    print(f"   -> {s['result']}")
    print()
print("Neither assumption in any pair is unreasonable. That is what makes these")
print("seams rather than mistakes - and why naming the handoff is the control.")

## 5 · Where it breaks — every function reports green

In [ ]:
def self_report(function):
    """Each function reports on the controls it operates. All true."""
    if function in STAKEHOLDERS:
        return {"function": function, "controls_operating": len(STAKEHOLDERS[function]["controls"]),
                "status": "green"}
    return {"function": function, "controls_operating": 0, "status": "n/a"}

for f in sorted(STAKEHOLDERS):
    r = self_report(f)
    print(f"   {r['function']:12s}{r['controls_operating']} controls  {r['status']}")
print()
print(f"functions reporting green : {len(STAKEHOLDERS)}/{len(STAKEHOLDERS)}")
print(f"open seams                : {len(SEAMS)}")
print()
print("A dashboard assembled from function self-reports is all green, and four")
print("material gaps are open. The dashboard is not lying - it is asking each")
print("function about the inside of its own box, and every failure here is")
print("between boxes.")
assert len(SEAMS) == 4

## 6 · The control — name the handoff, give it one owner

In [ ]:
HANDOFFS = {
 "trace retention schedule":      {"owner": "privacy",  "consumers": ["cyber", "legal"]},
 "tool scope in validation":      {"owner": "model_risk","consumers": ["cyber", "business_owner"]},
 "vendor no-train verification":  {"owner": "cyber",    "consumers": ["legal", "compliance"]},
 "re-tier on capability change":  {"owner": "compliance","consumers": ["business_owner", "cyber"]},
}
print(f"{'handoff artefact':32s}{'accountable':13s}consumers")
for h in sorted(HANDOFFS):
    v = HANDOFFS[h]
    print(f"{h:32s}{v['owner']:13s}{', '.join(v['consumers'])}")

covered = len(HANDOFFS)
print(f"\nseams: {len(SEAMS)}   handoffs with a named owner: {covered}")
print()
print("One artefact, many consumers, exactly one owner. The consumers matter as")
print("much as the owner: a handoff nobody consumes was never a handoff, and a")
print("handoff with two owners is the contested case from E1.0 again.")
assert covered == len(SEAMS)

## 7 · Verify — the two forgotten seats

In [ ]:
def governed(use_case):
    missing = [seat for seat in ("business_owner", "internal_audit")
               if seat not in use_case["seats"]]
    five = [f for f in STAKEHOLDERS if f in use_case["seats"]]
    return {"control_functions_present": len(five),
            "missing_seats": missing,
            "is_governed": not missing and len(five) == len(STAKEHOLDERS)}

CASES = [
 {"name": "customer support agent", "seats": list(STAKEHOLDERS) + ["business_owner", "internal_audit"]},
 {"name": "internal code assistant", "seats": list(STAKEHOLDERS)},
]
for c in CASES:
    g = governed(c)
    print(f"   {c['name']:26s}five functions: {g['control_functions_present']}/5   "
          f"missing: {g['missing_seats'] or 'none'}   governed: {g['is_governed']}")
print()
print("The second one has every control function at the table and no accountable")
print("owner. Five functions are governing something nobody has agreed to own,")
print("which is how a use case survives a review and still has no one to fund")
print("the remediation it was told to do.")
assert not governed(CASES[1])["is_governed"]

## What you just proved

Five stakeholder functions print with the question each is asking and the controls each operates — 22 controls in total. Four seam failures are shown as pairs of individually reasonable assumptions, and every function still self-reports green while all four gaps are open. Naming one accountable owner per handoff closes them, and a use case with all five control functions and no business owner is shown to be ungoverned.

## Your turn

Pick one of the four seams and find out, today, who owns it in your organisation. The answer 'I assume security does' from one side and 'I assume privacy does' from the other is the finding.

---

**Next → [E1.11 · Model risk management for AI systems](https://spbreed.github.io/cyber-commons/lessons/E1.11.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.10.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.10.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*